# 20장 · RAG 문서 로딩과 근거 메타데이터

같은 PDF에서 겉모습이 같은 두 쪽을 읽습니다. 문자층이 있는 1쪽에서는 반납 안내가 추출되고, 페이지 전체가 이미지인 2쪽에서는 빈 문자열이 나옵니다. 실제 파일에서 읽은 본문에 쪽 번호를 연결하고, 근거가 없는 주차 요금 질문과 추출에 실패한 쪽을 구분합니다.

실습 묶음 ZIP을 풀고 네 원본 파일과 같은 폴더에서 chapter20.ipynb를 여세요. 노트북만 내려받았다면 내려받은 노트북이 있는 폴더에서 여세요. 파일 이름을 바꿔도 됩니다. 첫 제공 셀이 노트북에 내장된 같은 원본을 임시 폴더에 복원합니다.

실행 환경은 Python 3.12.15, pypdf 6.21.4, langchain-core 1.6.12입니다. Jupyter에서 해당 커널을 선택하세요. 두 패키지가 없는 개인 커널에서는 새 셀에 %pip install pypdf==6.21.4 langchain-core==1.6.12를 실행하고 커널을 다시 시작하세요. 모델 API 키는 쓰지 않습니다.

셀 01에서 TXT의 문장을 읽고 셀 04에서 세 원문을 만듭니다. 셀 05·06은 같은 공지의 두 쪽, 과제 07은 추출 실패 처리입니다. 과제 09·10에서는 검색 결과를 나중에 검증할 기준 질문을 정합니다. 과제 11에서는 새 문서와 질문을 JSON으로 저장해 전달합니다.

### 제공 셀 · 실습 원본 네 파일 열기

이 셀을 실행하면 아래 실습에서 사용할 data_dir가 준비됩니다. 묶음을 풀었으면 그 폴더를 그대로 씁니다. 단독 노트북으로 실습할 때만 내장 사본을 복원합니다.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
from io import BytesIO
import base64, hashlib, json, zipfile

names = ("library.txt", "library-guide.pdf", "return-notice.pdf", "desk-change.txt")
data_dir = Path.cwd()
if not all((data_dir / name).is_file() for name in names):
    # 노트북만 내려받았을 때는 안에 보관한 원본을 임시 폴더에 엽니다.
    expected = "754f3750a4036db58caa719fc4d66391f6c98130be9a9371103032185730dfc3"
    candidates = sorted(Path.cwd().glob("*.ipynb"), key=lambda p: not p.name.startswith("chapter20"))
    for notebook_path in candidates:
        try:
            notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
            payload = notebook.get("metadata", {}).get("library_lesson_20_inputs_v1", "")
            if payload and hashlib.sha256(payload.encode("ascii")).hexdigest() == expected:
                break
        except (UnicodeError, ValueError, OSError):
            continue
    else:
        raise FileNotFoundError("수업 노트북을 작업 폴더에 두거나 실습 묶음의 파일 4개를 함께 여세요.")
    input_folder = TemporaryDirectory()
    data_dir = Path(input_folder.name)
    with zipfile.ZipFile(BytesIO(base64.b64decode(payload))) as archive:
        for name in names:
            (data_dir / name).write_bytes(archive.read(name))

print(", ".join(names))

library.txt, library-guide.pdf, return-notice.pdf, desk-change.txt


## TXT를 읽어 출처가 있는 문서 만들기

답변에서 “월요일”을 눌렀을 때 원문을 다시 열려면 파일 이름이 필요합니다. 먼저 파일 하나를 읽어 본문과 위치를 나눠 보세요.

### 자료를 미리 읽는 일과 질문을 받은 뒤 하는 일

대출 기간을 물으면 이용 안내 1쪽이 근거입니다. 이 장에서는 그 쪽을 다시 찾을 수 있도록 파일·쪽·본문을 한 건으로 저장합니다. 나중에 검색기를 평가할 때에는 질문에 적어 둔 기대 ID와 검색기가 실제로 돌려준 ID를 비교합니다.

### 셀 01 · TXT 본문과 파일 이름

library.txt를 열어 보세요. 가상 도서관의 휴관 안내 한 줄이 들어 있습니다. LangChain의 Document는 읽은 글을 page_content에, 파일 이름처럼 그 글을 찾을 단서를 metadata에 담습니다.

In [2]:
from langchain_core.documents import Document

txt_path = data_dir / "library.txt"
notice = Document(
    page_content=txt_path.read_text(encoding="utf-8").strip(),
    metadata={"source": txt_path.name, "page": None},
)
print(notice.page_content)
print(notice.metadata)

도서관은 월요일에 휴관합니다.
{'source': 'library.txt', 'page': None}


휴관일은 본문에 있고 원본 이름은 source에 있습니다. 이 TXT에는 쪽 구분이 없어 page를 None으로 남겼습니다. 공백을 정리할 때 strip()은 문장 안의 띄어쓰기를 건드리지 않습니다.

## PDF의 쪽 번호를 보존해 읽기

library-guide.pdf는 이 과정에서 쓰는 가상 도서관 이용 안내입니다. 실제 1쪽에는 대출 기간, 2쪽에는 휴관일, 3쪽에는 반납함 이용 시간이 있습니다. 21장에서 청크를 잘라도 원문으로 돌아올 수 있게 쪽 번호를 붙입니다.

### 셀 02 · PDF 2쪽에서 TXT와 같은 문장 찾기

PDF 뷰어에서 2쪽을 연 뒤 아래 값을 비교하세요. PdfReader의 pages는 0부터 시작합니다.

In [3]:
from pypdf import PdfReader

guide = data_dir / "library-guide.pdf"
reader = PdfReader(guide)
print(len(reader.pages))
print(reader.pages[1].extract_text().strip())

3
도서관은 월요일에 휴관합니다.


3은 전체 쪽수입니다. 인덱스 1에서 읽은 휴관 안내는 TXT와 문장이 같습니다. 본문 문자열만으로는 어느 파일을 인용했는지 구별할 수 없습니다.

TXT와 PDF에서 같은 휴관 안내 한 줄을 읽었습니다. 학생이 답변의 출처를 눌렀을 때 TXT는 파일을 열어 주면 되지만, 세 쪽짜리 PDF는 어느 쪽인지도 알려 줘야 합니다. 두 문서의 본문이 같다는 이유로 위치 정보까지 하나로 합치면 PDF의 2쪽을 되찾을 단서가 사라집니다.


### 셀 03 · 페이지마다 Document 만들기

아래 함수는 PDF 한 쪽을 한 건으로 읽습니다. 빈 쪽도 포함해 원래 순서와 위치를 돌려줍니다. 학생에게 보여 줄 쪽 번호는 첫 장부터 1로 셉니다.

In [4]:
def read_pages(path):
    pages = []
    for number, page in enumerate(PdfReader(path).pages, start=1):
        pages.append(Document(
            page_content=(page.extract_text() or "").strip(),
            metadata={"source": path.name, "page": number},
        ))
    return pages

pages = read_pages(guide)
print(pages[1].metadata)
print(pages[1].page_content)

{'source': 'library-guide.pdf', 'page': 2}
도서관은 월요일에 휴관합니다.


source가 library-guide.pdf이고 page가 2입니다. pages[1]로 꺼낸 뒤에도 metadata에 저장한 2를 그대로 인용합니다. 빈 페이지를 먼저 지우고 번호를 다시 매기면 이후 문장의 출처가 한 쪽씩 당겨질 수 있습니다.

### 셀 04 · 다음 장에 넘길 문서 묶음

세 쪽에 식별자를 붙여 21장의 청킹 입력을 만듭니다. 파일 이름은 pages에 남습니다. 다음 장에서 사용할 source에는 수강생이 읽을 자료명을 넣습니다.

In [5]:
def guide_record(page):
    number = page.metadata["page"]
    return {"id": f"library-{number}", "text": page.page_content,
            "source": "가상 도서관 이용 안내", "page": number}

documents = [guide_record(page) for page in pages]
print(documents[0])

{'id': 'library-1', 'text': '도서 대출 기간은 14일입니다.', 'source': '가상 도서관 이용 안내', 'page': 1}


library-1은 PDF 1쪽의 14일 규정입니다. 2쪽과 3쪽에는 각각 library-2와 library-3이 붙습니다. 다른 PDF까지 같은 접두사로 번호를 매기면 두 파일의 1쪽에 같은 ID가 생기므로 이 규칙은 이용 안내 한 파일에만 씁니다.

## 눈에는 같은 두 쪽에서 추출한 글 비교하기

return-notice.pdf는 비교용 파일입니다. 같은 안내문을 1쪽에는 선택할 수 있는 글자로, 2쪽에는 한 장의 스캔 이미지로 담았습니다. 이 파일의 쪽 번호와 이용 안내의 쪽 번호는 따로 읽습니다.

### 셀 05 · 1쪽에서 텍스트 추출하기

공지 PDF의 첫 페이지에서 보이는 제목과 문장을 읽어 보세요. 셀 출력에서 제목과 운영 문장이 각각 어느 줄에 있는지 확인하세요.

In [6]:
comparison = read_pages(data_dir / "return-notice.pdf")
print(comparison[0].page_content)

가상 도서관 반납 안내
무인 반납함은 24시간 이용할 수 있습니다.


제목 아래에 “무인 반납함은 24시간 이용할 수 있습니다.”가 나옵니다. 이 페이지에서 pypdf가 읽은 것은 PDF에 문자로 저장된 두 줄입니다.

### 셀 06 · 2쪽에서 빈 문자열 확인하기

PDF 뷰어로 본 2쪽과 추출 함수의 결과를 대조하세요. repr()은 빈 문자열을 따옴표 두 개로 표시합니다.

In [7]:
scan = comparison[1]
print(scan.metadata)
print(repr(scan.page_content))

{'source': 'return-notice.pdf', 'page': 2}
''


return-notice.pdf 2쪽에는 같은 공지가 보이지만 추출값은 빈 문자열입니다. 이 쪽은 글까지 픽셀로 저장했고 OCR 문자층도 없습니다. 따라서 이 파일에서는 원본 이미지를 문자로 인식하거나 텍스트 원본을 구해야 검색에 넣을 수 있습니다. 이미 OCR 문자층이 있는 스캔 파일이라면 결과가 달라질 수 있습니다.

#### 같은 공지인데 한 쪽만 빈 문자열인 이유

<mark>PDF 화면에 글자가 보여도 추출할 텍스트가 없을 수 있습니다.</mark> 1쪽과 2쪽의 추출값을 나란히 읽으세요. 빈 쪽은 자료 부재와 구별해 쪽 번호를 남겨야 나중에 OCR이나 수동 전사 대상으로 되돌릴 수 있습니다.

선수 개념 · 19장 LCEL 체인과 출력 파서

1쪽의 글자는 선택할 수 있고 2쪽에서는 공지 전체가 하나의 이미지로 보입니다. 셀 05와 06에서 같은 문장이 추출되는지 비교하세요.

### 운영 시간을 뒷받침하는 원문

답변에 ‘24시간’을 쓸 때 인용할 자료는 return-notice.pdf 1쪽의 운영 안내 문장입니다. 2쪽은 같은 문장이 보이지만 이번 추출 결과로는 검색할 수 없습니다.

### 과제 07 · 빈 쪽을 보류 목록으로 분리하기

split_pages(batch)를 작성하세요. 검색에 넣을 Document는 순서대로 첫 목록에, 본문이 없거나 공백뿐인 쪽은 원래 metadata만 두 번째 목록에 담습니다. 원래 2쪽을 보류해도 그 뒤의 쪽 번호는 바뀌지 않아야 합니다.

In [8]:
def split_pages(batch):
    # 검색에 넘길 Document 목록과 재확인할 위치 목록을 반환하세요.
    pass

# ready, review = split_pages(comparison)
# print([p.metadata["page"] for p in ready], review)

<details><summary>풀이와 실행 결과 보기</summary>
<pre><code>def split_pages(batch):
    ready, review = [], []
    for page in batch:
        if page.page_content.strip():
            ready.append(page)
        else:
            review.append(page.metadata.copy())
    return ready, review

ready, review = split_pages(comparison)
print([p.metadata[&quot;page&quot;] for p in ready], review)</code></pre>
<p>실행 결과</p><pre>[1] [{&#x27;source&#x27;: &#x27;return-notice.pdf&#x27;, &#x27;page&#x27;: 2}]</pre>
<p>1쪽이 남고 2쪽은 파일 이름과 함께 보류 목록에 들어갑니다. 보류 목록을 남겨야 2쪽이 검색 자료에서 빠진 이유를 원본에서 확인할 수 있습니다. 같은 안내가 문자로 저장된 1쪽은 검색 자료에 남습니다. 상태를 가르는 조건에서 strip()을 빼면 공백만 있는 쪽이 검색 입력으로 들어갑니다.</p>
</details>

## 문서로 답할 수 있는 질문과 없는 질문 구별하기

여기서부터는 이용 안내 세 쪽만 사용합니다. 질문의 정답을 아는 작성자가 기대 근거를 정해 두면 나중에 Retriever가 엉뚱한 쪽을 돌려줄 때 알아볼 수 있습니다. 실제 검색기는 23장, 근거를 넣어 답하는 체인은 24장에서 연결합니다.

### 셀 08 · 한 질문의 기대 ID를 원문으로 되짚기

개발 질문 dev-01에 붙인 library-1을 따라가세요. 금액이나 기간이 있는 답은 숫자까지 원문과 같은지 읽어야 합니다.

In [9]:
by_id = {doc["id"]: doc for doc in documents}
loan_case = {"question_id": "dev-01", "question": "대출 기간은 며칠인가요?",
             "expected_document_ids": ["library-1"], "answerable": True}
loan = by_id[loan_case["expected_document_ids"][0]]
print(loan["source"], loan["page"], loan["text"])

가상 도서관 이용 안내 1 도서 대출 기간은 14일입니다.


대출 기간을 물으면 이용 안내 1쪽이 근거입니다. 이 장에서는 그 쪽을 다시 찾을 수 있도록 파일·쪽·본문을 한 건으로 저장합니다. 나중에 검색기를 평가할 때에는 질문에 적어 둔 기대 ID와 검색기가 실제로 돌려준 ID를 비교합니다.

### 과제 09 · 잘못 붙인 근거를 찾아 고치기

아래 휴관일 질문은 존재하는 ID를 가리킵니다. 그 문서가 말하는 내용을 먼저 읽고 ID를 고치세요. 수정한 ID와 실제 PDF 쪽 번호, 원문 문장을 한 줄로 출력하세요.

In [10]:
closed_case = {"question_id": "dev-02", "question": "휴관 요일은 언제인가요?",
               "expected_document_ids": ["library-1"], "answerable": True}
# 잘못 연결한 원문을 조회한 다음, closed_case의 기대 ID를 바로잡으세요.

<details><summary>풀이와 실행 결과 보기</summary>
<pre><code>closed_case = {&quot;question_id&quot;: &quot;dev-02&quot;, &quot;question&quot;: &quot;휴관 요일은 언제인가요?&quot;,
               &quot;expected_document_ids&quot;: [&quot;library-1&quot;], &quot;answerable&quot;: True}
wrong = by_id[closed_case[&quot;expected_document_ids&quot;][0]]
print(wrong[&quot;text&quot;])
closed_case[&quot;expected_document_ids&quot;] = [&quot;library-2&quot;]
closed = by_id[closed_case[&quot;expected_document_ids&quot;][0]]
print(closed[&quot;id&quot;], closed[&quot;page&quot;], closed[&quot;text&quot;])</code></pre>
<p>실행 결과</p><pre>도서 대출 기간은 14일입니다.
library-2 2 도서관은 월요일에 휴관합니다.</pre>
<p>library-2를 원본 PDF 2쪽과 대조하면 월요일이라는 답을 확인할 수 있습니다. 코드로 확인한 ID의 존재 여부와 사람이 읽어 판단한 근거의 적합성은 별개의 검사입니다.</p>
</details>

### 과제 10 · 주차 요금 문항을 인계 목록에 추가하기

세 쪽을 읽어도 주차 요금을 확인할 수 없습니다. dev-03의 expected_document_ids와 answerable을 정하고, 이 문항을 답변 생성 단계로 보내도 될지 설명하세요. 아래 딕셔너리에서 두 개의 None을 고치세요.

In [11]:
parking_case = {"question_id": "dev-03", "question": "주차 요금은 얼마인가요?",
                "expected_document_ids": None, "answerable": None}
# print(parking_case["expected_document_ids"], parking_case["answerable"])

<details><summary>풀이와 실행 결과 보기</summary>
<pre><code>parking_case = {&quot;question_id&quot;: &quot;dev-03&quot;, &quot;question&quot;: &quot;주차 요금은 얼마인가요?&quot;,
                &quot;expected_document_ids&quot;: [], &quot;answerable&quot;: False}
print(parking_case[&quot;expected_document_ids&quot;], parking_case[&quot;answerable&quot;])</code></pre>
<p>실행 결과</p><pre>[] False</pre>
<p>[]는 인용할 원문이 없다는 표시입니다. False는 현재 문서 묶음에서 답을 확인할 수 없다는 뜻입니다. “무료”는 금액이 0원이라는 별도 주장이라 이 자료로 뒷받침할 수 없습니다.</p>
</details>

#### 질문과 근거가 만나는 곳

```mermaid
flowchart TD
nq["주차 질문"]
  nscope["자료 대조"]
  nlink["근거 연결"]
  nempty["근거 없음"]
  nq --> nscope
  nscope --> nlink
  nscope --> nempty
```

세 문서는 대출 기간·휴관일·반납함을 설명합니다. 주차 요금을 물었을 때 빈 기대 목록을 쓰는 이유는 자료 안에서 금액을 확인할 수 없기 때문입니다.

## 실습 · 다른 팀이 다시 열 수 있는 근거 묶음 만들기

도서관 안내에 분실물 접수 장소가 추가됐습니다. 새 자료를 읽을 사람에게 원문 파일과 질문을 건네고 같은 답을 찾는지 확인하세요. 기존 library-1부터 library-3까지는 그대로 두고 새 자료에는 다른 ID를 붙입니다.

### 과제 11 · 새 TXT와 질문을 JSON으로 인계하기

desk-change.txt는 처음 보는 인계 자료입니다. load_notice(path, doc_id)를 작성해 문서 ID·본문·파일 이름·쪽 번호를 반환하세요. “분실물은 어디에 맡기나요?”의 기대 ID와 “택배로도 받을 수 있나요?”의 기대 ID를 정합니다. 두 질문과 문서를 JSON 한 파일로 저장하고, 다시 읽은 파일에서 첫 질문의 원문 파일 이름과 근거 문장을 출력하세요. TXT에는 쪽 번호가 없습니다.

In [12]:
from pathlib import Path
import json

def load_notice(path, doc_id):
    # id, text, source, page 네 항목을 채우세요.
    pass

# 입력 파일: data_dir / "desk-change.txt"
# 문서 ID: desk-1
# 출력 파일: chapter20-output/desk-handoff.json

<details><summary>풀이와 실행 결과 보기</summary>
<pre><code>from pathlib import Path
import json

def load_notice(path, doc_id):
    return {&quot;id&quot;: doc_id, &quot;text&quot;: path.read_text(encoding=&quot;utf-8&quot;).strip(),
            &quot;source&quot;: path.name, &quot;page&quot;: None}

new_doc = load_notice(data_dir / &quot;desk-change.txt&quot;, &quot;desk-1&quot;)
questions = [
    {&quot;question&quot;: &quot;분실물은 어디에 맡기나요?&quot;, &quot;expected_document_ids&quot;: [&quot;desk-1&quot;], &quot;answerable&quot;: True},
    {&quot;question&quot;: &quot;택배로도 받을 수 있나요?&quot;, &quot;expected_document_ids&quot;: [], &quot;answerable&quot;: False},
]
payload = {&quot;documents&quot;: [new_doc], &quot;questions&quot;: questions}
output = Path(&quot;chapter20-output&quot;) / &quot;desk-handoff.json&quot;
output.parent.mkdir(exist_ok=True)
output.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding=&quot;utf-8&quot;)

received = json.loads(output.read_text(encoding=&quot;utf-8&quot;))
wanted = received[&quot;questions&quot;][0][&quot;expected_document_ids&quot;][0]
found = next(d for d in received[&quot;documents&quot;] if d[&quot;id&quot;] == wanted)
print(found[&quot;source&quot;], found[&quot;page&quot;])
print(found[&quot;text&quot;])</code></pre>
<p>실행 결과</p><pre>desk-change.txt None
분실물은 2층 안내 데스크에 맡겨 주세요.</pre>
<p>desk-change.txt와 안내 데스크 위치가 함께 나오면 상대 팀도 그 파일에서 답을 확인할 수 있습니다. 택배 질문에는 기대 ID가 없습니다. PDF 2쪽은 문자가 추출되지 않아 보류한 경우이고, 택배 질문은 원문을 모두 읽어도 답이 없는 경우입니다. 답안을 실행하면 현재 폴더 아래 chapter20-output에 인계 JSON이 생깁니다. 같은 이름으로 다시 실행하면 이 실습 파일을 갱신합니다.</p>
</details>

## 두 PDF에 모두 있는 ‘1쪽’ 구별하기

도서관 이용 안내와 반납 안내를 한 검색 목록에 넣습니다. 앞에서 읽은 두 PDF의 1쪽을 함께 넘기면 쪽 번호만으로 원문을 다시 열 수 있을까요?

### 셀 16 · 쪽 번호를 키로 쓴 색인에서 사라진 문서

두 PDF에서 각각 첫 페이지를 꺼내 하나의 사전에 넣습니다. 저장 직후에 남은 건수와 파일 이름을 확인하세요.

In [13]:
chosen_pages = [pages[0], comparison[0]]
by_number = {p.metadata["page"]: p for p in chosen_pages}
print(len(chosen_pages), len(by_number), by_number[1].metadata["source"])

2 1 return-notice.pdf


두 쪽을 읽었지만 사전에 남은 쪽은 하나입니다. 두 번째 대입에 사용한 키도 1이어서 반납 안내가 이용 안내를 덮어썼습니다. PDF 텍스트 추출은 두 번 모두 성공했습니다.

### 과제 17 · 출처와 쪽 번호로 원문 찾아가기

(파일 이름, 쪽 번호)를 키로 두 페이지를 다시 색인하세요. 두 항목이 남았는지 확인한 뒤 반납 안내 1쪽의 마지막 문장을 읽으세요.

In [14]:
# chosen_pages를 돌면서 각 Document의 source와 page를 한 쌍으로 묶으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>page_index = {(p.metadata[&quot;source&quot;], p.metadata[&quot;page&quot;]): p for p in chosen_pages}
return_page = page_index[(&quot;return-notice.pdf&quot;, 1)]
print(len(page_index))
print(return_page.page_content.splitlines()[-1])</code></pre>
<p>실행 결과</p><pre><code>2
무인 반납함은 24시간 이용할 수 있습니다.</code></pre>
<p>2와 무인 반납함 이용 문장이 나옵니다. 파일 이름까지 키에 넣었으므로 서로 다른 자료의 같은 쪽수를 함께 보관할 수 있습니다. 여기서는 두 파일의 이름이 서로 다릅니다.</p>
</details>

<mark>쪽 번호는 해당 문서 안에서만 위치를 가리킵니다.</mark> 답변에 1쪽이라고 쓰려면 어느 파일의 1쪽인지도 함께 저장해야 합니다.

다음 장에서 한 페이지를 여러 청크로 나누어도 이 두 값은 원래 페이지를 가리킵니다.

### 인계 전 원본 대조

셀 04의 documents를 펼쳐 이용 안내 1·2·3쪽에 대출 14일, 월요일 휴관, 반납함 24시간이 차례로 연결되는지 확인하세요. 동료에게 dev-02의 원문과 쪽수, dev-03을 보류한 이유를 설명해 보세요. 동료는 질문별 기대 ID만 보고 같은 PDF 위치를 찾을 수 있어야 합니다.

짝 실습에서는 상대 팀이 새로 쓴 짧은 TXT를 desk-change.txt 대신 읽고 인계용 JSON의 질문도 바꿔 보세요. 받는 팀은 원본과 JSON만 보고 각 기대 ID의 근거 문장을 찾아 작성자에게 돌려줍니다. 질문에 답할 수 없으면 어느 파일들을 확인했는지 적습니다.

연습을 마친 뒤 새 질문 “반납함은 몇 시간 이용할 수 있나요?”와 “카페의 메뉴를 알려 주세요.”에 답할 근거가 있는지 각각 확인하세요. 첫 질문은 library-3, 둘째는 빈 목록과 대조합니다. 이 공개 연습 문항으로 실제 검색기의 정답률을 추정할 수는 없습니다. 다음 장에서는 library-1의 문장을 잘게 나눈 뒤 원문 쪽수를 다시 찾아갑니다.

## 참고 문서

- [pypdf · PDF 텍스트 추출과 스캔 이미지](https://pypdf.readthedocs.io/en/6.21.4/user/extract-text.html)
- [LangChain · Document의 page_content와 metadata](https://reference.langchain.com/python/langchain-core/documents/base/Document)